In [6]:
# Dataset download
# Your manual specifies Retail_Sales_Data.csv, but does not provide a direct download URL. You can use this alternative:
# - Kaggle – Customer Shopping Trends / Retail Sales Dataset: https://www.kaggle.com/datasets?utm_source=chatgpt.com
# - Retail Sales Dataset – GitHub search: https://github.com/search?q=Retail_Sales_Data.csv&type=code&utm_source=chatgpt.com




# ============================================================
# EXPERIMENT NO. 6
# DATA AGGREGATION
# Analyzing Sales Performance by Region in a Retail Company
# Dataset: Retail_Sales_Data.csv
# ============================================================

# STEP 1: Import required libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files

# STEP 2: Upload the retail sales dataset

print("Upload Retail_Sales_Data.csv")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]

# STEP 3: Load the dataset

df = pd.read_csv(file_name)

print("\n========== ORIGINAL DATASET ==========")
print(df.head())

# STEP 4: Explore the dataset

print("\nDataset Shape:")
print(df.shape)

print("\nDataset Information:")
df.info()

print("\nStatistical Description:")
print(df.describe(include="all"))

# STEP 5: Clean column names

# Convert column names to lowercase and replace spaces
# and special characters with underscores.

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

print("\nCleaned Column Names:")
print(df.columns.tolist())

# STEP 6: Identify important columns automatically

def find_column(possible_names):

    for name in possible_names:
        if name in df.columns:
            return name

    return None


region_col = find_column([
    "region", "sales_region", "area", "location"
])

sales_col = find_column([
    "sales", "sales_amount", "total_sales",
    "amount", "total_amount", "revenue", "price"
])

category_col = find_column([
    "category", "product_category",
    "product_line", "product"
])

mall_col = find_column([
    "shopping_mall", "mall", "store",
    "store_name", "branch"
])

quantity_col = find_column([
    "quantity", "quantity_sold", "units_sold"
])

date_col = find_column([
    "date", "transaction_date", "order_date"
])

print("\n========== DETECTED COLUMNS ==========")
print("Region:", region_col)
print("Sales:", sales_col)
print("Product Category:", category_col)
print("Shopping Mall:", mall_col)
print("Quantity:", quantity_col)
print("Date:", date_col)

# STEP 7: Handle missing values and duplicates

print("\nMissing Values:")
print(df.isnull().sum())

# Remove duplicate rows.

df = df.drop_duplicates()

# Fill missing categorical values with mode.

categorical_columns = df.select_dtypes(
    include=["object", "string"]
).columns

for column in categorical_columns:

    mode_value = df[column].mode()

    if not mode_value.empty:
        df[column] = df[column].fillna(mode_value[0])
    else:
        df[column] = df[column].fillna("Unknown")

# STEP 8: Convert numerical columns into numeric format

if sales_col is not None:

    df[sales_col] = pd.to_numeric(
        df[sales_col].astype(str).str.replace(
            r"[$,]", "", regex=True
        ),
        errors="coerce"
    )

if quantity_col is not None:

    df[quantity_col] = pd.to_numeric(
        df[quantity_col],
        errors="coerce"
    )

# Remove rows where sales amount is missing.

if sales_col is not None:
    df = df.dropna(subset=[sales_col])

print("\nCleaned Dataset:")
print(df.head())

# STEP 9: Display total sales

if sales_col is not None:

    total_sales = df[sales_col].sum()

    print("\n========== TOTAL SALES ==========")
    print("Total Sales Amount:", total_sales)

# STEP 10: Group sales by region

if region_col is not None and sales_col is not None:

    region_sales = (
        df.groupby(region_col)[sales_col]
        .sum()
        .sort_values(ascending=False)
    )

    print("\n========== TOTAL SALES BY REGION ==========")
    print(region_sales)

    # Identify the top-performing region.

    top_region = region_sales.idxmax()
    top_sales = region_sales.max()

    print("\nTop Performing Region:", top_region)
    print("Total Sales:", top_sales)

    # STEP 11: Bar chart of sales by region

    plt.figure(figsize=(10, 5))

    plt.bar(
        region_sales.index.astype(str),
        region_sales.values
    )

    plt.title("Total Sales by Region")
    plt.xlabel("Region")
    plt.ylabel("Total Sales Amount")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

    # STEP 12: Pie chart of sales distribution by region

    plt.figure(figsize=(8, 8))

    plt.pie(
        region_sales.values,
        labels=region_sales.index.astype(str),
        autopct="%1.1f%%",
        startangle=90
    )

    plt.title("Sales Distribution by Region")
    plt.show()

else:
    print("\nRegion or sales column not found.")
    region_sales = None

# STEP 13: Group sales by product category

if category_col is not None and sales_col is not None:

    category_sales = (
        df.groupby(category_col)[sales_col]
        .sum()
        .sort_values(ascending=False)
    )

    print("\n========== SALES BY PRODUCT CATEGORY ==========")
    print(category_sales)

    # Bar chart for product categories.

    plt.figure(figsize=(10, 5))

    plt.bar(
        category_sales.index.astype(str),
        category_sales.values
    )

    plt.title("Total Sales by Product Category")
    plt.xlabel("Product Category")
    plt.ylabel("Total Sales Amount")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# STEP 14: Group sales by region and product category

if (
    region_col is not None
    and category_col is not None
    and sales_col is not None
):

    region_category_sales = pd.pivot_table(
        df,
        values=sales_col,
        index=region_col,
        columns=category_col,
        aggfunc="sum",
        fill_value=0
    )

    print("\n========== REGION AND CATEGORY SALES ==========")
    print(region_category_sales)

    # Grouped bar chart.

    region_category_sales.plot(
        kind="bar",
        figsize=(12, 6)
    )

    plt.title("Sales by Region and Product Category")
    plt.xlabel("Region")
    plt.ylabel("Total Sales")
    plt.xticks(rotation=45)
    plt.legend(title="Product Category")
    plt.tight_layout()
    plt.show()

# STEP 15: Group sales by shopping mall

if mall_col is not None and sales_col is not None:

    mall_sales = (
        df.groupby(mall_col)[sales_col]
        .sum()
        .sort_values(ascending=False)
    )

    print("\n========== SALES BY SHOPPING MALL ==========")
    print(mall_sales)

    plt.figure(figsize=(12, 5))

    plt.bar(
        mall_sales.index.astype(str),
        mall_sales.values
    )

    plt.title("Total Sales by Shopping Mall")
    plt.xlabel("Shopping Mall")
    plt.ylabel("Total Sales")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# STEP 16: Calculate quantity sold by region

if region_col is not None and quantity_col is not None:

    region_quantity = (
        df.groupby(region_col)[quantity_col]
        .sum()
        .sort_values(ascending=False)
    )

    print("\n========== QUANTITY SOLD BY REGION ==========")
    print(region_quantity)

# STEP 17: Calculate summary statistics

if sales_col is not None:

    print("\n========== SALES SUMMARY STATISTICS ==========")

    print("Total Sales:", df[sales_col].sum())
    print("Average Sales:", df[sales_col].mean())
    print("Median Sales:", df[sales_col].median())
    print("Maximum Sales:", df[sales_col].max())
    print("Minimum Sales:", df[sales_col].min())

# STEP 18: Save aggregated results

if region_col is not None and sales_col is not None:

    region_sales.to_csv("region_sales_summary.csv")

if category_col is not None and sales_col is not None:

    category_sales.to_csv("category_sales_summary.csv")

if (
    region_col is not None
    and category_col is not None
    and sales_col is not None
):

    region_category_sales.to_csv(
        "region_category_sales.csv"
    )

# Save the cleaned dataset.

df.to_csv("cleaned_retail_sales.csv", index=False)

print("\n========== EXPERIMENT COMPLETED ==========")
print("Retail sales aggregation completed successfully.")
print("Summary files have been saved.")

# STEP 19: Download the cleaned dataset

files.download("cleaned_retail_sales.csv")


Upload Retail_Sales_Data.csv


KeyboardInterrupt: 